# AnthropicToolCallState

```lua
stateDiagram-v2
    INIT --> CHAT
    CHAT --> FINAL
```

```mermaid
stateDiagram-v2
    Direction LR
    INIT --> CHAT
    CHAT --> FINAL
```


### a) Normal Chat Flow

In [1]:
from gai.asm import AsyncStateMachine

with AsyncStateMachine.StateMachineBuilder(
    """
    INIT --> CHAT
    CHAT--> FINAL
    """
) as builder:
    fsm = builder.build(
        {
            "INIT": {
                "input_data": {
                    "llm_config": {"type": "getter", "dependency": "get_llm_config"},
                }
            },
            "CHAT": {
                "module_path": "gai.asm.states",
                "class_name": "AnthropicChatState",
                "title": "CHAT",
                "input_data": {
                    "llm_config": {"type": "state_bag", "dependency": "llm_config"},
                },
                "output_data": ["streamer", "get_assistant_message"],
            },
            "FINAL": {
                "output_data": ["monologue"],
            },
        },
        get_llm_config=lambda state: {
            "client_type": "anthropic",
            #"model": "claude-opus-4-20250514",
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 32000,
            "temperature": 0.7,
            "top_p": 0.95,
        }
    )

## Step 2: INIT --> TOOL_CALL

fsm.user_message = "Tell me a one sentence story."
await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    if (isinstance(chunk,str)):
        print(chunk, end='', flush=True)
print("\n\n")

## Step 4: Print the state history
for message in fsm.state_history[1]["output"]["monologue"].list_messages():
    print(
        f"{message.header.timestamp} {message.header.sender} > {message.body.content}"
    )

The last person on Earth sat alone in a room, when suddenly there was a knock at the door.



1752650902.8756385 User > Tell me a one sentence story.
1752650904.7781334 Assistant > [{'citations': None, 'text': 'The last person on Earth sat alone in a room, when suddenly there was a knock at the door.', 'type': 'text'}]


### b) With Dialogue Recap

In [1]:
# Create an artificial dialogue history for testing
import os
from gai.messages import Dialogue, MessagePydantic

messages = [
    MessagePydantic(
        **{
            "id": "b1e5f98c-f6eb-47de-a6e2-387510d970f9",
            "header": {
                "sender": "User",
                "recipient": "Sara",
                "timestamp": 1751308157.270983,
                "order": 0,
            },
            "body": {
                "type": "chat.send",
                "dialogue_id": "00000000-0000-0000-0000-000000000000",
                "round_no": 0,
                "step_no": 0,
                "role": "user",
                "content": "I love horror stories, are you familiar with them?",
            },
        }
    ),
    MessagePydantic(
        **{
            "id": "abbc7961-45dc-4973-aaf4-a6224ed35d37",
            "header": {
                "sender": "Sara",
                "recipient": "User",
                "timestamp": 1751308167.3488164,
                "order": 1,
            },
            "body": {
                "type": "chat.reply",
                "dialogue_id": "00000000-0000-0000-0000-000000000000",
                "round_no": 0,
                "step_no": 1,
                "chunk_no": 10,
                "chunk": "<eom>",
                "role": "assistant",
                "content": "Yes, I am familiar with horror stories. They are a fascinating genre that can evoke strong emotions and create a sense of suspense and fear. Do you have any specific horror stories in mind that you would like to discuss?",
            },
        }
    ),
]
from gai.lib.constants import DEFAULT_GUID
from gai.messages import Dialogue
dialogue = Dialogue(messages=messages)
recap = dialogue.extract_recap()

#---

from gai.asm import AsyncStateMachine

with AsyncStateMachine.StateMachineBuilder(
    """
    INIT --> CHAT
    CHAT--> FINAL
    """
) as builder:
    fsm = builder.build(
        {
            "INIT": {
                "input_data": {
                    "llm_config": {"type": "getter", "dependency": "get_llm_config"},
                    "recap": {"type":"getter","dependency":"get_recap"}
                }
            },
            "CHAT": {
                "module_path": "gai.asm.states",
                "class_name": "AnthropicChatState",
                "title": "CHAT",
                "input_data": {
                    "llm_config": {
                        "type": "state_bag", 
                        "dependency": "llm_config"
                    },
                    "recap": {
                        "type": "state_bag",
                        "dependency": "recap",
                    },
                },
                "output_data": ["streamer", "get_assistant_message"],
            },
            "FINAL": {
                "output_data": ["monologue"],
            },
        },
        get_llm_config=lambda state: {
            "client_type": "anthropic",
            # "model": "claude-opus-4-20250514",
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 32000,
            "temperature": 0.7,
            "top_p": 0.95,
        },
        get_recap=lambda state: recap,
    )

## Step 2: INIT --> TOOL_CALL

fsm.user_message = "Tell me a one sentence story."
await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    if isinstance(chunk, str):
        print(chunk, end="", flush=True)
print("\n\n")

## Step 4: Print the state history
for message in fsm.state_history[1]["output"]["monologue"].list_messages():
    print(
        f"{message.header.timestamp} {message.header.sender} > {message.body.content}"
    )

dialogue.list_messages()

The last thing she heard before the basement door slammed shut was her own voice from upstairs, calling her name.



1752651746.7898405 User > 
            Here is a recap of the conversation:
            User: I love horror stories, are you familiar with them?
Sara: Yes, I am familiar with horror stories. They are a fascinating genre that can evoke strong emotions and create a sense of suspense and fear. Do you have any specific horror stories in mind that you would like to discuss?
            
            You may respond to my following message using the context you have learnt.
            Tell me a one sentence story.
            
1752651752.3513026 Assistant > [{'citations': None, 'text': 'The last thing she heard before the basement door slammed shut was her own voice from upstairs, calling her name.', 'type': 'text'}]


[MessagePydantic(id='b1e5f98c-f6eb-47de-a6e2-387510d970f9', header=MessageHeaderPydantic(sender='User', recipient='Sara', timestamp=1751308157.270983, order=0), body=ChatSendBodyPydantic(type='chat.send', dialogue_id='00000000-0000-0000-0000-000000000000', round_no=0, step_no=0, message_id='00000000-0000-0000-0000-000000000000.1', content_type='text', role='user', content='I love horror stories, are you familiar with them?')),
 MessagePydantic(id='abbc7961-45dc-4973-aaf4-a6224ed35d37', header=MessageHeaderPydantic(sender='Sara', recipient='User', timestamp=1751308167.3488164, order=1), body=ChatReplyBodyPydantic(type='chat.reply', dialogue_id='00000000-0000-0000-0000-000000000000', round_no=0, step_no=1, message_id='00000000-0000-0000-0000-000000000000.2', chunk_no=10, chunk='<eom>', content_type='text', role='assistant', content='Yes, I am familiar with horror stories. They are a fascinating genre that can evoke strong emotions and create a sense of suspense and fear. Do you have any 

### b) Tool Call Flow

In [2]:
from gai.asm import AsyncStateMachine
from gai.mcp.client import McpAggregatedClient

with AsyncStateMachine.StateMachineBuilder(
    """
    INIT --> CHAT
    CHAT--> FINAL
    """
) as builder:
    fsm = builder.build(
        {
            "INIT": {
                "input_data": {
                    "llm_config": {"type": "getter", "dependency": "get_llm_config"},
                    "mcp_client": {"type": "getter", "dependency": "get_mcp_client"},
                }
            },
            "CHAT": {
                "module_path": "gai.asm.states",
                "class_name": "AnthropicChatState",
                "title": "CHAT",
                "input_data": {
                    "llm_config": {"type": "state_bag", "dependency": "llm_config"},
                    "mcp_client": {"type": "state_bag","dependency": "mcp_client"},
                },
                "output_data": ["streamer", "get_assistant_message"],
            },
            "FINAL": {
                "output_data": ["monologue"],
            },
        },
        get_llm_config=lambda state: {
            "client_type": "anthropic",
            #"model": "claude-opus-4-20250514",
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 32000,
            "temperature": 0.7,
            "top_p": 0.95,
        },
        get_mcp_client=lambda state: McpAggregatedClient(["mcp-time"])
    )

## Step 2: INIT --> CHAT

fsm.user_message = "What time is it in Singapore?"
await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    if (isinstance(chunk,str)):
        print(chunk, end='', flush=True)
print("\n\n")

## Step 4: Print the state history
for message in fsm.state_history[1]["output"]["monologue"].list_messages():
    print(
        f"{message.header.timestamp} {message.header.sender} > {message.body.content}"
    )

I'll get the current time in Singapore for you.



1752650920.4678345 User > What time is it in Singapore?
1752650922.5959585 Assistant > [{'citations': None, 'text': "I'll get the current time in Singapore for you.", 'type': 'text'}, {'id': 'toolu_01LAQS9QKoqC2PML28mkZxxR', 'input': {'format': 'YYYY-MM-DD HH:mm:ss', 'timezone': 'Asia/Singapore'}, 'name': 'current_time', 'type': 'tool_use'}]
